<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Фактуры:</h2>

----

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public class LineItem
{
    private string _name;
    private decimal _price;
    public string Name
    {
        get => _name;
        set => _name = string.IsNullOrWhiteSpace(value) ? "Без названия" : value;
    }

    public decimal Price
    {
        get => _price;
        set => _price = value < 0 ? 0 : value;
    }

    public LineItem(string name, decimal price)
    {
        Name = name;
        Price = price;
    }
}

public class Invoice
{
    private string _invoiceNumber;
    private decimal _totalAmount;

    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set => _invoiceNumber = string.IsNullOrWhiteSpace(value) ? "INV-000" : value;
    }

    public DateTime IssueDate { get; set; }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set => _totalAmount = value < 0 ? 0 : value;
    }

    protected List<LineItem> Items = new List<LineItem>();

    public Invoice() : this("INV-DEFAULT", DateTime.Now) { }

    public Invoice(string number, DateTime date)
    {
        InvoiceNumber = number;
        IssueDate = date;
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = Items.Sum(item => item.Price);
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        Items.Add(lineItem);
        Console.WriteLine($"Добавлена позиция в [{InvoiceNumber}]: {lineItem.Name} ({lineItem.Price} руб.)");
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (Items.Remove(lineItem))
        {
            Console.WriteLine($"Удалена позиция из [{InvoiceNumber}]: {lineItem.Name}");
        }
    }

    public virtual void TransferItemTo(LineItem item, Invoice targetInvoice)
    {
        if (targetInvoice == null) return;

        if (Items.Contains(item))
        {
            this.RemoveLine(item);
            targetInvoice.AddLine(item);
            Console.WriteLine($"--> [Взаимодействие]: Позиция '{item.Name}' перенесена из счета {this.InvoiceNumber} в счет {targetInvoice.InvoiceNumber}");
        }
        else
        {
            Console.WriteLine($"--> [Ошибка]: Позиция '{item.Name}' не найдена в счете {InvoiceNumber}");
        }
    }

    public virtual void MergeWith(Invoice otherInvoice)
    {
        if (otherInvoice == null || otherInvoice == this) return;

        Console.WriteLine($"\n--> [Взаимодействие]: Объединение счета {otherInvoice.InvoiceNumber} со счетом {this.InvoiceNumber}...");
        
        var itemsToTransfer = new List<LineItem>(otherInvoice.Items);
        foreach (var item in itemsToTransfer)
        {
            otherInvoice.TransferItemTo(item, this);
        }
    }
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string number, DateTime date, DateTime supplyDate) 
        : base(number, date)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"   * Товар будет поставлен: {SupplyDate.ToShortDateString()}");
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }

    public ServiceInvoice(string number, DateTime date, DateTime serviceDate) 
        : base(number, date)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("   * Причина удаления: Услуга была отменена клиентом.");
    }
}

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }

    public CombinedInvoice(string number, DateTime date, bool returnAllowed) 
        : base(number, date)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal baseTotal = base.CalculateTotal();
        if (ReturnAllowed)
        {
            TotalAmount = baseTotal + (baseTotal * 0.05m); // Сбор 5%
        }
        return TotalAmount;
    }
}

Console.WriteLine("=== 1. Создание товарного счета ===");
GoodsInvoice goodsInvoice = new GoodsInvoice("Т-001", DateTime.Now, DateTime.Now.AddDays(3));
LineItem laptop = new LineItem("Ноутбук", 50000);
LineItem mouse = new LineItem("Мышь", 1500);
goodsInvoice.AddLine(laptop);
goodsInvoice.AddLine(mouse);

Console.WriteLine("\n=== 2. Создание услугового счета ===");
ServiceInvoice serviceInvoice = new ServiceInvoice("У-001", DateTime.Now, DateTime.Now);
LineItem setup = new LineItem("Настройка ПК", 3000);
serviceInvoice.AddLine(setup);

Console.WriteLine("\n=== 3. ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: Перенос одной позиции ===");
goodsInvoice.TransferItemTo(mouse, serviceInvoice);

Console.WriteLine("\n=== 4. ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: Объединение счетов ===");
CombinedInvoice combinedInvoice = new CombinedInvoice("К-001", DateTime.Now, true);
combinedInvoice.MergeWith(serviceInvoice);

Console.WriteLine("\n=== 5. Расчет итогов после всех взаимодействий ===");
Console.WriteLine($"Сумма товарного счета ({goodsInvoice.InvoiceNumber}): {goodsInvoice.CalculateTotal()} руб.");
Console.WriteLine($"Сумма услугового счета ({serviceInvoice.InvoiceNumber}): {serviceInvoice.CalculateTotal()} руб.");
Console.WriteLine($"Сумма комбинированного счета ({combinedInvoice.InvoiceNumber}): {combinedInvoice.CalculateTotal()} руб. (с учетом 5% сбора)");


=== 1. Создание товарного счета ===
Добавлена позиция в [Т-001]: Ноутбук (50000 руб.)
   * Товар будет поставлен: 23.09.2026
Добавлена позиция в [Т-001]: Мышь (1500 руб.)
   * Товар будет поставлен: 23.09.2026

=== 2. Создание услугового счета ===
Добавлена позиция в [У-001]: Настройка ПК (3000 руб.)

=== 3. ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: Перенос одной позиции ===
Удалена позиция из [Т-001]: Мышь
Добавлена позиция в [У-001]: Мышь (1500 руб.)
--> [Взаимодействие]: Позиция 'Мышь' перенесена из счета Т-001 в счет У-001

=== 4. ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: Объединение счетов ===

--> [Взаимодействие]: Объединение счета У-001 со счетом К-001...
Удалена позиция из [У-001]: Настройка ПК
   * Причина удаления: Услуга была отменена клиентом.
Добавлена позиция в [К-001]: Настройка ПК (3000 руб.)
--> [Взаимодействие]: Позиция 'Настройка ПК' перенесена из счета У-001 в счет К-001
Удалена позиция из [У-001]: Мышь
   * Причина удаления: Услуга была отменена клиентом.
Добавлена позиция в [К-001]: Мышь (1500 ру